# Day 1: Infrastructure Foundation & Iceberg Storage Setup

This notebook simulates the local environment for Apache Iceberg over DuckDB and PySpark to validate the Bronze raw payload schema with audit metadata.

In [ ]:
!pip install pyspark pyiceberg duckdb splink pydeequ -q

In [ ]:
import os
from pyspark.sql import SparkSession

# Set up Spark Session with Iceberg configurations
spark = SparkSession.builder \
    .appName("Day01_Storage_Setup") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session initialized with Iceberg catalog.")

In [ ]:
# Create the Bronze raw payload schema with audit metadata
create_table_sql = """
CREATE TABLE IF NOT EXISTS local.db.bronze_raw_payload (
    ingestion_timestamp TIMESTAMP,
    source_system_id STRING,
    payload_id STRING,
    raw_payload_json STRING
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
"""

spark.sql("CREATE DATABASE IF NOT EXISTS local.db")
spark.sql(create_table_sql)

print("Bronze Iceberg table created successfully.")

In [ ]:
# Verify the schema
spark.sql("DESCRIBE TABLE local.db.bronze_raw_payload").show(truncate=False)